# V3 Stage 3: Causal Mechanistic Experiments

Thin notebook — all logic lives in `stage3_causal.py`.
Edit CONFIG, run all cells.

**Experiments:**
- 3A: Attribution patching (gradient-based head ranking)
- 3B: Targeted activation patching (causal validation)
- 3C: Logit lens under ablation (trajectory change)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    "model": "Qwen/Qwen2.5-3B-Instruct",
    "stage2_path": "results/Qwen2.5-3B-Instruct/stage2_logit_lens_20260303_005011.json",
    "point": (5, 5),        # (keys, updates) or None for auto-pick from stage2
    "trials": 50,
    "top_k": 20,            # top K heads from attribution patching
    "gpu": 0,
    "experiments": ["3A", "3B", "3C"],  # which to run
}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SETUP
# ═══════════════════════════════════════════════════════════════════════════

import sys, os
from pathlib import Path

notebook_dir = Path(os.getcwd()).resolve()
if notebook_dir.name == "v3":
    PROJECT_ROOT = notebook_dir.parent
else:
    for p in [notebook_dir, notebook_dir.parent, notebook_dir.parent.parent]:
        if (p / "mechanistic_probing_v2" / "core").exists():
            PROJECT_ROOT = p
            break
    else:
        raise RuntimeError("Cannot find project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "v3") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "v3"))

# Resolve stage2 path relative to v3/
s2_path = CONFIG["stage2_path"]
if not Path(s2_path).is_absolute():
    CONFIG["stage2_path"] = str(PROJECT_ROOT / "v3" / s2_path)

print(f"Project root: {PROJECT_ROOT}")
print(f"Stage 2 path: {CONFIG['stage2_path']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOAD MODEL
# ═══════════════════════════════════════════════════════════════════════════

from mechanistic_probing_v2.core.model_loader import load_model

model, tokenizer, info = load_model(
    CONFIG["model"],
    gpu_idx=CONFIG.get("gpu"),
)
print(f"\nReady: {info.n_layers} layers, {info.n_heads} heads, d_model={info.d_model}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ANALYZE STAGE 2 (preview before running)
# ═══════════════════════════════════════════════════════════════════════════

from stage3_causal import analyze_stage2

s2_info = analyze_stage2(CONFIG["stage2_path"])
print(f"Best operating point: {s2_info['best_point']}")
print()
for pk, pi in s2_info["point_info"].items():
    print(f"  {pk}: peak_layer={pi['peak_layer']}, suppression={pi['suppression']:.4f}, "
          f"concentrated={pi['concentrated']}, n_failures={pi['n_failures']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN STAGE 3
# ═══════════════════════════════════════════════════════════════════════════

from stage3_causal import run_stage3

results = run_stage3(CONFIG, model=model, tokenizer=tokenizer, info=info)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ATTRIBUTION HEATMAP (Exp 3A)
# ═══════════════════════════════════════════════════════════════════════════

import numpy as np
import matplotlib.pyplot as plt

if "3A" in results["experiments"]:
    attr = np.array(results["experiments"]["3A"]["attribution_scores"])
    
    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(attr.T, aspect="auto", cmap="hot")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Head")
    ax.set_title(f"Attribution Patching: Head Importance for P(v_last) — {CONFIG['model'].split('/')[-1]}")
    plt.colorbar(im, ax=ax, label="Attribution score")
    
    # Mark top-5 heads
    for h in results["experiments"]["3A"]["top_heads"][:5]:
        ax.plot(h["layer"], h["head"], "wo", markersize=10, markeredgewidth=2)
        ax.text(h["layer"] + 0.3, h["head"], h["label"], color="white", fontsize=8)
    
    # Mark critical layers from Stage 2
    s2_pt = results.get("stage2_info", {})
    if "critical_layers" in s2_pt:
        for L in s2_pt["critical_layers"][:4]:
            ax.axvline(x=L, color="cyan", linewidth=0.5, alpha=0.5)
    
    plt.tight_layout()
    plt.show()
else:
    print("Exp 3A not run")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PATCHING EFFECT BAR CHART (Exp 3B)
# ═══════════════════════════════════════════════════════════════════════════

if "3B" in results["experiments"]:
    hr = results["experiments"]["3B"]["head_results"]
    sorted_heads = sorted(hr.items(), key=lambda x: -x[1]["mean_delta_p_last"])
    
    # Top 20
    top = sorted_heads[:20]
    labels = [h[0] for h in top]
    deltas = [h[1]["mean_delta_p_last"] for h in top]
    stds = [h[1]["std_delta_p_last"] for h in top]
    
    # Color by set membership
    target_info = {h["label"]: h for h in results.get("target_heads", {}).get("heads", [])}
    colors = []
    for label in labels:
        ti = target_info.get(label, {})
        if ti.get("in_A") and ti.get("in_B"):
            colors.append("#4CAF50")  # both sets — green
        elif ti.get("in_A"):
            colors.append("#2196F3")  # Set A only — blue
        else:
            colors.append("#FF9800")  # Set B only — orange
    
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.barh(range(len(top)), deltas, xerr=stds, color=colors, edgecolor="white")
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(labels)
    ax.set_xlabel("ΔP(v_last) when head is patched (clean → corrupted)")
    ax.set_title("Exp 3B: Targeted Patching Effect")
    ax.axvline(x=0, color="black", linewidth=0.5)
    ax.invert_yaxis()
    
    # Legend
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color="#4CAF50", label="Set A ∩ B (attribution + critical layer)"),
        Patch(color="#2196F3", label="Set A only (attribution)"),
        Patch(color="#FF9800", label="Set B only (critical layer)"),
    ], loc="lower right", fontsize=9)
    
    plt.tight_layout()
    plt.show()
else:
    print("Exp 3B not run")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ABLATION TRAJECTORY COMPARISON (Exp 3C)
# ═══════════════════════════════════════════════════════════════════════════

if "3C" in results["experiments"]:
    r3c = results["experiments"]["3C"]
    normal = np.array(r3c["avg_normal_trajectory"])
    ablated = np.array(r3c["avg_ablated_trajectory"])
    n_values, n_layers = normal.shape
    correct_idx = n_values - 1
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Left: P(v_last) normal vs ablated
    ax = axes[0]
    ax.plot(range(n_layers), normal[correct_idx, :], "r-", linewidth=2, label="Normal")
    ax.plot(range(n_layers), ablated[correct_idx, :], "g--", linewidth=2, label="Heads ablated")
    ax.set_xlabel("Layer")
    ax.set_ylabel("P(v_last)")
    ax.set_title(f"P(v_last) trajectory: normal vs ablated")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Right: all values, zoomed to active region
    ax = axes[1]
    active_start = max(0, n_layers - 10)
    layers = range(active_start, n_layers)
    
    for vi in range(n_values):
        color = "#4CAF50" if vi == correct_idx else "#9E9E9E"
        lw = 2 if vi == correct_idx else 0.8
        ax.plot(layers, normal[vi, active_start:], color=color, linewidth=lw,
                linestyle="-", alpha=0.7)
        ax.plot(layers, ablated[vi, active_start:], color=color, linewidth=lw,
                linestyle="--", alpha=0.7)
    
    ax.set_xlabel("Layer")
    ax.set_ylabel("P(value)")
    ax.set_title(f"All values (solid=normal, dashed=ablated)")
    ax.grid(True, alpha=0.3)
    
    fig.suptitle(f"Exp 3C: Ablating {', '.join(r3c['heads_ablated'])}", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print(f"\nP(v_last) at final layer:")
    print(f"  Normal:  {r3c['p_last_normal_final']:.4f}")
    print(f"  Ablated: {r3c['p_last_ablated_final']:.4f}")
    print(f"  Delta:   {r3c['p_last_ablated_final'] - r3c['p_last_normal_final']:+.4f}")
else:
    print("Exp 3C not run")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONVERGENCE CHECK: Do Set A and Set B agree?
# ═══════════════════════════════════════════════════════════════════════════

if "3A" in results["experiments"] and "3B" in results["experiments"]:
    top_attr = results["experiments"]["3A"]["top_heads"][:20]
    critical = set(results["stage2_info"]["critical_layers"])
    
    in_critical = sum(1 for h in top_attr if h["layer"] in critical)
    outside = [h for h in top_attr if h["layer"] not in critical]
    
    print(f"Convergence check:")
    print(f"  Attribution top-20: {in_critical} in critical layers, {len(outside)} outside")
    
    if outside:
        print(f"\n  Heads OUTSIDE critical layers (mechanistically interesting):")
        for h in outside:
            print(f"    {h['label']}: attribution={h['attribution']:.4f}")
    
    # Check patching agreement
    hr = results["experiments"]["3B"]["head_results"]
    sorted_patch = sorted(hr.items(), key=lambda x: -x[1]["mean_delta_p_last"])[:10]
    top_attr_labels = {h["label"] for h in top_attr[:10]}
    top_patch_labels = {h[0] for h in sorted_patch}
    overlap = top_attr_labels & top_patch_labels
    
    print(f"\n  Top-10 attribution vs top-10 patching overlap: {len(overlap)}/10")
    print(f"    Shared: {overlap}")
else:
    print("Need both 3A and 3B to check convergence")